In [3]:
import os
from fastapi import FastAPI, File, UploadFile
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from PIL import Image
import numpy as np

In [4]:
MODEL_PATH = 'mnist_model.h5'
app = FastAPI()

In [ ]:
if not os.path.exists(MODEL_PATH):
    print("Training model...")
    (x_train, y_train), _ = mnist.load_data()
    x_train = x_train / 255.0

    model = Sequential([
        Flatten(input_shape=(28,28)),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=1)  
    model.save(MODEL_PATH)
else:
    model = load_model(MODEL_PATH)

Training model...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


d:\VAmazinumCamp2025\jupyter\lesson_30_task\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9266 - loss: 0.2556


In [7]:
@app.get("/")
def root():
    return {"message": "MNIST CV API is running"}

@app.post("/predict/")
async def predict(file: UploadFile = File(...)):
    img = Image.open(file.file).convert('L').resize((28,28))
    img_array = np.array(img) / 255.0
    img_array = img_array.reshape(1,28,28)
    prediction = model.predict(img_array)
    predicted_digit = int(np.argmax(prediction))
    return {"predicted_digit": predicted_digit}